In [1]:
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.axes_grid1 import make_axes_locatable
from icecream import ic
import jax
import jax.numpy as np
from memo import memo
from enum import IntEnum
from ipywidgets import interact, fixed
import ipywidgets as widgets
import pandas as pd

In [4]:
# type:ignore

K_values = 10
alpha_idx = np.arange(K_values)
alphas = np.linspace(0, 1, alpha_idx.size)

beta_idx = np.arange(K_values)
betas = np.linspace(0, 2, beta_idx.size)

delta_idx = np.arange(K_values)
deltas = np.linspace(0, 1, delta_idx.size)

gamma_idx = np.arange(K_values)
gammas = np.linspace(0, 10, gamma_idx.size)

endowment_idx = np.arange(K_values)
endowments = np.linspace(1, 20, endowment_idx.size)

class A(IntEnum):
    Low=0,
    Medium=1,
    High=2,
    Lots=3,
    All=4


@jax.jit
def payoffs(a):
    return np.array([.0, .25, .5, .75, 1.0])[a]


@jax.jit
def rho_beta(beta, mu, sd):
    return jax.scipy.stats.norm.pdf(beta, mu, sd)


@jax.jit
def get_prior(param, mean, sd):
    return jax.scipy.stats.norm.pdf(param, mean, sd)


@memo
def Alice_gives_Bob[a:A, alpha:alphas, beta:betas, delta:deltas, gamma:gammas](endowment_true, 
                                                                               endowment_believed, 
                                                                               endowment_sd,
                                                                               pres_target,
                                                                               pres_sd):

    Alice: knows(alpha, beta, delta, gamma)

    Alice: thinks [
        Bob: thinks [
            Alice: given(endowment in endowments, wpp=get_prior(endowment, endowment_believed, endowment_sd)),
            Alice: given(alpha in alphas, wpp=1),
            Alice: given(beta in betas, wpp=1),
            Alice: chooses(a in A, wpp=exp(
                alpha * (1 - payoffs(a)) * endowment
                + beta * payoffs(a) * endowment
            )),

            # Presentational goal of A for C
            pres: given(beta in betas, wpp=rho_beta(beta, pres_target, pres_sd)),
            pres: given(endowment in endowments, wpp=get_prior(endowment, endowment_believed, endowment_sd)),
        ],
    ]

    Alice: chooses (a in A, wpp=exp(
        alpha * (1 - payoffs(a)) * endowment_true
        + beta * payoffs(a) * endowment_true
        + imagine [
            Bob: observes [Alice.a] is a,
            - delta * Bob [ KL [ Alice.beta | pres.beta ]] # Alice wants Bob to think Alice's beta is close to rho_beta
            #+ delta * log( Bob [ Pr[ Alice.beta >= pres_target ]]) # Alice wants Bob to think Alice's beta is close to rho_beta
            + gamma * log( Bob [ Pr[ Alice.endowment <= (endowment_believed - 1.96 * endowment_sd) ]] ) # Alice wants Bob to think Alice's endowment is lower than the mean of the prior
        ]
    ))

    return Pr [ Alice.a == a, Alice.alpha == alpha,  Alice.beta == beta, Alice.delta == delta, Alice.gamma == gamma]


In [5]:
endowment_true = 20
endowment_believed = 5
endowment_sd = 1
pres_target = 1
pres_sd = 1.5

out = Alice_gives_Bob(endowment_true, endowment_believed, endowment_sd, pres_target, pres_sd)[:, -1, :, -1, -1]
out_norm = out / np.sum(out, axis=0, keepdims=True)

out_norm.round(3)

Array([[0.75900006, 0.50200003, 0.23400001, 0.075     , 0.016     ,
        0.002     , 0.        , 0.        , 0.        , 0.        ],
       [0.23700002, 0.476     , 0.674     , 0.65500003, 0.432     ,
        0.19700001, 0.066     , 0.017     , 0.003     , 0.        ],
       [0.003     , 0.021     , 0.09100001, 0.268     , 0.53800005,
        0.744     , 0.75900006, 0.577     , 0.316     , 0.12900001],
       [0.        , 0.        , 0.        , 0.002     , 0.013     ,
        0.056     , 0.17500001, 0.404     , 0.67      , 0.82900006],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.002     , 0.011     , 0.042     ]],      dtype=float32)

In [6]:
out_norm = out / np.sum(out, axis=1, keepdims=True)

out_norm.round(3)

Array([[0.47800002, 0.316     , 0.147     , 0.047     , 0.01      ,
        0.002     , 0.        , 0.        , 0.        , 0.        ],
       [0.086     , 0.17300001, 0.245     , 0.23700002, 0.157     ,
        0.071     , 0.024     , 0.006     , 0.001     , 0.        ],
       [0.001     , 0.006     , 0.026     , 0.078     , 0.156     ,
        0.216     , 0.22000001, 0.16800001, 0.09200001, 0.037     ],
       [0.        , 0.        , 0.        , 0.001     , 0.006     ,
        0.026     , 0.081     , 0.18800001, 0.312     , 0.386     ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.001     , 0.006     , 0.04      , 0.2       , 0.753     ]],      dtype=float32)

## Visualise simulations

In [7]:
# Create custom widgets that display both value and name
# Create custom widgets with sufficient width for descriptions
def create_labeled_dropdown(options, description):
    # Convert options to display both value and name
    labeled_options = [(f"{option.name} ({option.value})", option) for option in options]
    
    # Create dropdown with style to accommodate longer descriptions
    dropdown = widgets.Dropdown(
        options=labeled_options, 
        description=description,
        style={'description_width': 'auto'}  # This allows description to use needed space
    )
    
    # Set layout to provide more space
    dropdown.layout.width = '50%'  # Use full width
    
    return dropdown

# Create your custom widgets
endowment_widget = widgets.FloatSlider(min=1, max=20, step=1, value=20, description="Alice's true endowment", style={'description_width': 'auto'})
endowment_belief_widget = widgets.FloatSlider(min=0, max=20, step=1, value=5, description="Bob's Belief of Alice's endowment", style={'description_width': 'auto'})
endowment_sd_widget = widgets.FloatSlider(min=0.2, max=5, step=.2, value=2, description="- Bob's Belief SD", style={'description_width': 'auto'})
pres_target_widget = widgets.FloatSlider(min=0, max=1, step=0.1, value=1, description="Alice's target for Bob's pres", style={'description_width': 'auto'})
pres_sd_widget = widgets.FloatSlider(min=0.05, max=1, step=.05, value=.15, description="- Target SD", style={'description_width': 'auto'})

gamma_widget = widgets.FloatSlider(min=0, max=20, step=0.5, value=10, description="Alice's need to hide endowment", style={'description_width': 'auto'})

endowment_widget.layout.width = '50%'
endowment_belief_widget.layout.width = '50%'
endowment_sd_widget.layout.width = '50%'
pres_target_widget.layout.width = '50%'
pres_sd_widget.layout.width = '50%'
gamma_widget.layout.width = '50%'

# Use interactive instead of interact
def show_interactive(
        endowment_true,
        endowment_believed,
        endowment_sd,
        pres_target,
        pres_sd,
        gamma,
    ):

    gamma_idx = np.argmin(np.abs(gammas - gamma))
    alpha = alphas[-1]
    delta = deltas[-1]

    
    fig, axes = plt.subplots(1, 4, figsize=(14, 5))

    ax = axes[0]
    ax.sharex(axes[1])
    ax.sharey(axes[1])

    ax.set_title(f"Observer's inferences \n $\\delta=1$, $\\gamma={gamma}$", size=14, rotation=0)
    p_action_params = Alice_gives_Bob(endowment_true, endowment_believed, endowment_sd, pres_target, pres_sd)[:, -1, :, -1, gamma_idx]
    p_action_params = p_action_params / np.sum(p_action_params, axis=1, keepdims=True)
    betas_means_obs = p_action_params @ betas

    sns.heatmap(
        p_action_params.T,
        square=True if K_values == len(A) else False,
        annot=True if K_values <= 5 else False,
        fmt=".2f",
        cmap="Blues",
        vmin=0,
        vmax=1,
        yticklabels=[f'{beta:.2f}' for beta in betas],
        xticklabels=[f'{a.name}\n{(endowment_true * payoffs(a)):.0f}' for a in A],
        cbar=False,
        ax=ax
    )
    ax.set_ylabel("$p(\\beta|a)$", size=13, rotation=90)
    ax.set_xlabel("Alice's action", size=13)
    ax.tick_params(axis='y', labelsize=10, labelrotation=0)
    ax.tick_params(axis='x', labelsize=10, labelrotation=0)
    #ax.set_yticklabels([], rotation=45, ha='right')
    

    ax = axes[1]
    ax.set_title(f"Bob's inferences \n $\\delta=1$, $\\gamma=0$", size=14)
    p_action_params = Alice_gives_Bob(endowment_true, endowment_believed, endowment_sd, pres_target, pres_sd)[:, -1, :, -1, 0]
    p_action_params = p_action_params / np.sum(p_action_params, axis=1, keepdims=True)
    betas_means_bob = p_action_params @ betas
    im = sns.heatmap(
        p_action_params.T,
        square=True if K_values == len(A) else False,
        annot=True if K_values <= 5 else False,
        fmt=".2f",
        cmap="Reds",
        vmin=0,
        vmax=1,
        yticklabels=[f'{beta:.2f}' for beta in betas],
        xticklabels=[f'{a.name}\n{(endowment_true * payoffs(a)):.0f}' for a in A],
        cbar=False,
        ax=ax
    )
    #ax.set_ylabel("$\\beta$", size=13)
    
    ax.tick_params(axis='y', labelsize=10, labelrotation=0)
    ax.tick_params(axis='x', labelsize=10, labelrotation=0)
    ax.set_xlabel("Alice's action", size=13)

    ax = axes[2]
    ax.sharey(axes[2])
    ax.sharex(axes[2])
    ax.set_title(f"Bob's naive inferences \n $\\delta=0$, $\\gamma=0$", size=14)
    p_action_params = Alice_gives_Bob(endowment_true, endowment_believed, endowment_sd, pres_target, pres_sd)[:, -1, :, 0, 0]
    p_action_params = p_action_params / np.sum(p_action_params, axis=1, keepdims=True)
    betas_means_bob_naive = p_action_params @ betas
    im = sns.heatmap(
        p_action_params.T,
        square=True if K_values == len(A) else False,
        annot=True if K_values <= 5 else False,
        fmt=".2f",
        cmap="Greens",
        vmin=0,
        vmax=1,
        yticklabels=[f'{beta:.2f}' for beta in betas],
        xticklabels=[f'{a.name}\n{(endowment_true * payoffs(a)):.0f}' for a in A],
        cbar=False,
        ax=ax
    )
    #ax.set_ylabel("$\\beta$", size=13)
    
    ax.tick_params(axis='y', labelsize=10, labelrotation=0)
    ax.tick_params(axis='x', labelsize=10, labelrotation=0)
    ax.set_xlabel("Alice's action", size=13)

    # Create colorbar on the right side without affecting subplot sizes
    #divider = make_axes_locatable(axes[-1])
    #cax = divider.append_axes("right", size="7%", pad=0.0)
    #plt.colorbar(im.collections[0], cax=cax, label='Posterior Probability')

    ax = axes[3]
    ax.set_title(f"Posterior mean $\\beta$", size=14)

    # Plot the posterior means for each action
    df_means = pd.DataFrame({
        'Action': [a.name for a in A],
        'Posterior Mean Beta': betas_means_obs,
        'Posterior Mean Beta Bob': betas_means_bob,
        'Posterior Mean Beta Bob Naive': betas_means_bob_naive
    })
    df_means = df_means.melt(id_vars='Action', value_vars=df_means.columns[1:], var_name='Type', value_name='Mean Beta')
    palette = sns.color_palette("Blues", n_colors=1) + sns.color_palette("Reds", n_colors=1) + sns.color_palette("Greens", n_colors=1)
    sns.barplot(x='Action', y='Mean Beta', hue='Type', data=df_means, ax=ax, palette=palette)
    ax.set_ylabel("$E[\\beta|a]$", size=13)
    # Put legend at the bottom of the plot
    # Get the handles and labels from the current axes
    handles, labels = ax.get_legend_handles_labels()
    new_labels = ['Observer', 'Bob', 'Bob Naive']
    ax.legend(handles, new_labels, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, fontsize=11)

    sns.despine(ax=ax)

    fig.suptitle(f"Alice's action probabilities\n Endowment: {endowment_true}, Bob's belief: ${endowment_believed}\\pm {endowment_sd}$,  $\\rho(\\beta) = N({pres_target}, {pres_sd})$\n", size=14)
    plt.tight_layout()
    plt.show()

widgets.interactive(
    show_interactive,
    endowment_true=endowment_widget,
    endowment_believed=endowment_belief_widget,
    endowment_sd=endowment_sd_widget,
    pres_target=pres_target_widget,
    pres_sd=pres_sd_widget,
    gamma=gamma_widget
)

interactive(children=(FloatSlider(value=20.0, description="Alice's true endowment", layout=Layout(width='50%')…

In [8]:
# Create custom widgets that display both value and name
# Create custom widgets with sufficient width for descriptions
def create_labeled_dropdown(options, description):
    # Convert options to display both value and name
    labeled_options = [(f"{option.name} ({option.value})", option) for option in options]
    
    # Create dropdown with style to accommodate longer descriptions
    dropdown = widgets.Dropdown(
        options=labeled_options, 
        description=description,
        style={'description_width': 'auto'}  # This allows description to use needed space
    )
    
    # Set layout to provide more space
    dropdown.layout.width = '50%'  # Use full width
    
    return dropdown

# Create your custom widgets
endowment_widget = widgets.FloatSlider(min=1, max=20, step=1, value=20, description="Alice's true endowment", style={'description_width': 'auto'})
endowment_belief_widget = widgets.FloatSlider(min=0, max=20, step=1, value=5, description="Bob's Belief of Alice's endowment", style={'description_width': 'auto'})
endowment_sd_widget = widgets.FloatSlider(min=0.2, max=5, step=.2, value=2, description="- Bob's Belief SD", style={'description_width': 'auto'})
pres_target_widget = widgets.FloatSlider(min=0, max=1, step=0.1, value=1, description="Alice's target for Bob's pres", style={'description_width': 'auto'})
pres_sd_widget = widgets.FloatSlider(min=0.05, max=1, step=.05, value=.15, description="- Target SD", style={'description_width': 'auto'})

gamma_widget = widgets.FloatSlider(min=0, max=20, step=0.5, value=10, description="Alice's need to hide endowment", style={'description_width': 'auto'})

endowment_widget.layout.width = '50%'
endowment_belief_widget.layout.width = '50%'
endowment_sd_widget.layout.width = '50%'
pres_target_widget.layout.width = '50%'
pres_sd_widget.layout.width = '50%'
gamma_widget.layout.width = '50%'

# Use interactive instead of interact
def show_interactive(
        endowment_true,
        endowment_believed,
        endowment_sd,
        pres_target,
        pres_sd,
        gamma,
    ):

    gamma_idx = np.argmin(np.abs(gammas - gamma))
    alpha = alphas[-1]
    delta = deltas[-1]

    
    fig, axes = plt.subplots(2, 4, figsize=(14, 8))

    ax = axes[0, 0]
    ax.sharex(axes[0, 1])
    ax.sharey(axes[0, 1])

    ax.set_title(f"Observer's inferences \n $\\delta=1$, $\\gamma={gamma}$", size=14, rotation=0)
    p_action_params = Alice_gives_Bob(endowment_true, endowment_believed, endowment_sd, pres_target, pres_sd)[:, -1, :, -1, gamma_idx]
    p_action_params = p_action_params / np.sum(p_action_params, axis=1, keepdims=True)
    betas_means_obs = p_action_params @ betas

    sns.heatmap(
        p_action_params.T,
        square=True if K_values == len(A) else False,
        annot=True if K_values <= 5 else False,
        fmt=".2f",
        cmap="Blues",
        vmin=0,
        vmax=1,
        yticklabels=[f'{beta:.2f}' for beta in betas],
        xticklabels=[f'{a.name}\n{(endowment_true * payoffs(a)):.0f}' for a in A],
        cbar=False,
        ax=ax
    )
    ax.set_ylabel("$p(\\beta|a)$", size=13, rotation=90)
    ax.set_xlabel("Alice's action", size=13)
    ax.tick_params(axis='y', labelsize=10, labelrotation=0)
    ax.tick_params(axis='x', labelsize=10, labelrotation=0)
    #ax.set_yticklabels([], rotation=45, ha='right')

    ax = axes[1, 0]
    ax.sharex(axes[1, 0])
    ax.sharey(axes[1, 0])

    ax.set_title(f"Observer's inferences \n $\\delta=1$, $\\gamma={gamma}$", size=14, rotation=0)
    p_action_params = Alice_gives_Bob(endowment_true, endowment_believed, endowment_sd, pres_target, pres_sd)[:, -1, :, -1, gamma_idx]
    p_action_params = p_action_params / np.sum(p_action_params, axis=1, keepdims=True)
    betas_means_obs = p_action_params @ betas

    sns.heatmap(
        p_action_params.T,
        square=True if K_values == len(A) else False,
        annot=True if K_values <= 5 else False,
        fmt=".2f",
        cmap="Blues",
        vmin=0,
        vmax=1,
        yticklabels=[f'{beta:.2f}' for beta in betas],
        xticklabels=[f'{a.name}\n{(endowment_true * payoffs(a)):.0f}' for a in A],
        cbar=False,
        ax=ax
    )
    ax.set_ylabel("$p(\\beta|a)$", size=13, rotation=90)
    ax.set_xlabel("Alice's action", size=13)
    ax.tick_params(axis='y', labelsize=10, labelrotation=0)
    ax.tick_params(axis='x', labelsize=10, labelrotation=0)
    

    # COLUMN 2
    ax = axes[0, 1]
    ax.set_title(f"Bob's inferences \n $\\delta=1$, $\\gamma=0$", size=14)
    p_action_params = Alice_gives_Bob(endowment_true, endowment_believed, endowment_sd, pres_target, pres_sd)[:, -1, :, -1, 0]
    p_action_params = p_action_params / np.sum(p_action_params, axis=1, keepdims=True)
    betas_means_bob = p_action_params @ betas
    im = sns.heatmap(
        p_action_params.T,
        square=True if K_values == len(A) else False,
        annot=True if K_values <= 5 else False,
        fmt=".2f",
        cmap="Reds",
        vmin=0,
        vmax=1,
        yticklabels=[f'{beta:.2f}' for beta in betas],
        xticklabels=[f'{a.name}\n{(endowment_true * payoffs(a)):.0f}' for a in A],
        cbar=False,
        ax=ax
    )
    #ax.set_ylabel("$\\beta$", size=13)
    
    ax.tick_params(axis='y', labelsize=10, labelrotation=0)
    ax.tick_params(axis='x', labelsize=10, labelrotation=0)
    ax.set_xlabel("Alice's action", size=13)

    ax = axes[0, 2]
    ax.sharey(axes[0, 2])
    ax.sharex(axes[0, 2])
    ax.set_title(f"Bob's naive inferences \n $\\delta=0$, $\\gamma=0$", size=14)
    p_action_params = Alice_gives_Bob(endowment_true, endowment_believed, endowment_sd, pres_target, pres_sd)[:, -1, :, 0, 0]
    p_action_params = p_action_params / np.sum(p_action_params, axis=1, keepdims=True)
    betas_means_bob_naive = p_action_params @ betas
    im = sns.heatmap(
        p_action_params.T,
        square=True if K_values == len(A) else False,
        annot=True if K_values <= 5 else False,
        fmt=".2f",
        cmap="Greens",
        vmin=0,
        vmax=1,
        yticklabels=[f'{beta:.2f}' for beta in betas],
        xticklabels=[f'{a.name}\n{(endowment_true * payoffs(a)):.0f}' for a in A],
        cbar=False,
        ax=ax
    )
    #ax.set_ylabel("$\\beta$", size=13)
    
    ax.tick_params(axis='y', labelsize=10, labelrotation=0)
    ax.tick_params(axis='x', labelsize=10, labelrotation=0)
    ax.set_xlabel("Alice's action", size=13)

    # Create colorbar on the right side without affecting subplot sizes
    #divider = make_axes_locatable(axes[-1])
    #cax = divider.append_axes("right", size="7%", pad=0.0)
    #plt.colorbar(im.collections[0], cax=cax, label='Posterior Probability')

    ax = axes[0, 3]
    ax.set_title(f"Posterior mean $\\beta$", size=14)

    # Plot the posterior means for each action
    df_means = pd.DataFrame({
        'Action': [a.name for a in A],
        'Posterior Mean Beta': betas_means_obs,
        'Posterior Mean Beta Bob': betas_means_bob,
        'Posterior Mean Beta Bob Naive': betas_means_bob_naive
    })
    df_means = df_means.melt(id_vars='Action', value_vars=df_means.columns[1:], var_name='Type', value_name='Mean Beta')
    palette = sns.color_palette("Blues", n_colors=1) + sns.color_palette("Reds", n_colors=1) + sns.color_palette("Greens", n_colors=1)
    sns.barplot(x='Action', y='Mean Beta', hue='Type', data=df_means, ax=ax, palette=palette)
    ax.set_ylabel("$E[\\beta|a]$", size=13)
    # Put legend at the bottom of the plot
    # Get the handles and labels from the current axes
    handles, labels = ax.get_legend_handles_labels()
    new_labels = ['Observer', 'Bob', 'Bob Naive']
    ax.legend(handles, new_labels, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, fontsize=11)

    sns.despine(ax=ax)

    fig.suptitle(f"Alice's action probabilities\n Endowment: {endowment_true}, Bob's belief: ${endowment_believed}\\pm {endowment_sd}$,  $\\rho(\\beta) = N({pres_target}, {pres_sd})$\n", size=14)
    plt.tight_layout()
    plt.show()

widgets.interactive(
    show_interactive,
    endowment_true=endowment_widget,
    endowment_believed=endowment_belief_widget,
    endowment_sd=endowment_sd_widget,
    pres_target=pres_target_widget,
    pres_sd=pres_sd_widget,
    gamma=gamma_widget
)

interactive(children=(FloatSlider(value=20.0, description="Alice's true endowment", layout=Layout(width='50%')…

In [ ]:
# type:ignore

K_values = 10
alpha_idx = np.arange(K_values)
alphas = np.linspace(0, 1, alpha_idx.size)

beta_idx = np.arange(K_values)
betas = np.linspace(0, 2, beta_idx.size)

delta_idx = np.arange(K_values)
deltas = np.linspace(0, 1, delta_idx.size)

gamma_idx = np.arange(K_values)
gammas = np.linspace(0, 10, gamma_idx.size)

endowment_idx = np.arange(K_values)
endowments = np.linspace(1, 20, endowment_idx.size)

class Endowment(IntEnum): 
    Poor=0
    Rich=1

class A(IntEnum):
    Low=0,
    Medium=1,
    High=2,
    Lots=3,
    All=4

@jax.jit
def get_endowment_value(endowment):
    return np.array([1, 20])[endowment]

@jax.jit
def payoffs(a):
    return np.array([.0, .25, .5, .75, 1.0])[a]

@jax.jit
def get_endowment_likelihood(endowment, a, b):
    return jax.scipy.stats.beta.pdf(endowment, 1 + a, 1 + b)

@jax.jit
def rho_beta(beta, mu, sd):
    return jax.scipy.stats.norm.pdf(beta, mu, sd)

@jax.jit
def get_prior(param, mean, sd):
    return jax.scipy.stats.norm.pdf(param, mean, sd)


@memo
def Alice_gives_Bob[a:A, alpha:alphas, beta:betas, delta:deltas](endowment_true, 
                                                                 endowment_believed, 
                                                                 endowment_sd,
                                                                 pres_target,
                                                                 pres_sd):

    Alice: knows(alpha, beta, delta)

    Alice: thinks [
        Bob: thinks [
            Alice: given(endowment in Endowment, wpp=get_prior(endowment, endowment_believed, endowment_sd)),

            Alice: given(beta in betas, wpp=1),
            Alice: chooses(a in A, wpp=exp(
                (1 - payoffs(a)) * endowment
                + beta * payoffs(a) * endowment
            )),

            # Presentational goal of A for C
            Alice: given(beta_pres in betas, wpp=rho_beta(beta_pres, pres_target, pres_sd)),
        ],
    ]

    Alice: chooses (a in A, wpp=exp(
        alpha * (1 - payoffs(a)) * endowment_true
        + beta * payoffs(a) * endowment_true
        + imagine [
            Bob: observes [Alice.a] is a,
            - delta * Bob [ KL [ Alice.beta | Alice.beta_pres ]] # Alice wants Bob to think Alice's beta is close to rho_beta
        ]
    ))

    return Pr [ Alice.a == a, Alice.alpha == alpha,  Alice.beta == beta, Alice.delta == delta]


In [21]:
Endowment.Poor

<Endowment.Poor: 10>

In [20]:
get_endowment_likelihood(0.1, 2, 2)

Array(0.24300006, dtype=float32, weak_type=True)

In [17]:
for endowment in Endowment:
    print(get_prior(endowment, endowment_believed, endowment_sd))

1.4867194e-06
0.0


In [15]:
# Create custom widgets that display both value and name
# Create custom widgets with sufficient width for descriptions
def create_labeled_dropdown(options, description):
    # Convert options to display both value and name
    labeled_options = [(f"{option.name} ({option.value})", option) for option in options]
    
    # Create dropdown with style to accommodate longer descriptions
    dropdown = widgets.Dropdown(
        options=labeled_options, 
        description=description,
        style={'description_width': 'auto'}  # This allows description to use needed space
    )
    
    # Set layout to provide more space
    dropdown.layout.width = '50%'  # Use full width
    
    return dropdown

# Create your custom widgets
endowment_widget = widgets.FloatSlider(min=1, max=20, step=1, value=20, description="Alice's true endowment", style={'description_width': 'auto'})
endowment_belief_widget = widgets.FloatSlider(min=0, max=20, step=1, value=11, description="Bob's Belief of Alice's endowment", style={'description_width': 'auto'})
endowment_sd_widget = widgets.FloatSlider(min=0.2, max=5, step=.2, value=2, description="- Bob's Belief SD", style={'description_width': 'auto'})
pres_target_widget = widgets.FloatSlider(min=0, max=2, step=0.1, value=3, description="Alice's target for Bob's pres", style={'description_width': 'auto'})
pres_sd_widget = widgets.FloatSlider(min=0.05, max=1, step=.05, value=1, description="- Target SD", style={'description_width': 'auto'})

gamma_widget = widgets.FloatSlider(min=0, max=20, step=0.5, value=10, description="Alice's need to hide endowment", style={'description_width': 'auto'})

endowment_widget.layout.width = '50%'
endowment_belief_widget.layout.width = '50%'
endowment_sd_widget.layout.width = '50%'
pres_target_widget.layout.width = '50%'
pres_sd_widget.layout.width = '50%'
gamma_widget.layout.width = '50%'

# Use interactive instead of interact
def show_interactive(
        endowment_true,
        endowment_believed,
        endowment_sd,
        pres_target,
        pres_sd,
        gamma,
    ):

    gamma_idx = np.argmin(np.abs(gammas - gamma))
    alpha = alphas[-1]
    delta = deltas[-1]

    
    fig, axes = plt.subplots(1, 4, figsize=(14, 5))

    ax = axes[0]
    ax.sharex(axes[1])
    ax.sharey(axes[1])

    ax.set_title(f"Observer's inferences \n $\\delta=1$, $\\gamma={gamma}$", size=14, rotation=0)
    p_action_params = Alice_gives_Bob(endowment_true, endowment_believed, endowment_sd, pres_target, pres_sd)[:, -1, :, -1]
    p_action_params = p_action_params / np.sum(p_action_params, axis=1, keepdims=True)
    betas_means_obs = p_action_params @ betas

    sns.heatmap(
        p_action_params.T,
        square=True if K_values == len(A) else False,
        annot=True if K_values <= 5 else False,
        fmt=".2f",
        cmap="Blues",
        vmin=0,
        vmax=1,
        yticklabels=[f'{beta:.2f}' for beta in betas],
        xticklabels=[f'{a.name}\n{(endowment_true * payoffs(a)):.0f}' for a in A],
        cbar=False,
        ax=ax
    )
    ax.set_ylabel("$p(\\beta|a)$", size=13, rotation=90)
    ax.set_xlabel("Alice's action", size=13)
    ax.tick_params(axis='y', labelsize=10, labelrotation=0)
    ax.tick_params(axis='x', labelsize=10, labelrotation=0)
    #ax.set_yticklabels([], rotation=45, ha='right')
    

    ax = axes[1]
    ax.set_title(f"Bob's inferences \n $\\delta=1$", size=14)
    p_action_params = Alice_gives_Bob(endowment_true, endowment_believed, endowment_sd, pres_target, pres_sd)[:, -1, :, -1]
    p_action_params = p_action_params / np.sum(p_action_params, axis=0, keepdims=True)
    betas_means_bob = p_action_params @ betas
    im = sns.heatmap(
        p_action_params.T,
        square=True if K_values == len(A) else False,
        annot=True if K_values <= 5 else False,
        fmt=".2f",
        cmap="Reds",
        vmin=0,
        vmax=1,
        yticklabels=[f'{beta:.2f}' for beta in betas],
        xticklabels=[f'{a.name}\n{(endowment_true * payoffs(a)):.0f}' for a in A],
        cbar=False,
        ax=ax
    )
    #ax.set_ylabel("$\\beta$", size=13)
    
    ax.tick_params(axis='y', labelsize=10, labelrotation=0)
    ax.tick_params(axis='x', labelsize=10, labelrotation=0)
    ax.set_xlabel("Alice's action", size=13)

    ax = axes[2]
    ax.sharey(axes[2])
    ax.sharex(axes[2])
    ax.set_title(f"Bob's naive inferences \n $\\delta=0$, $\\gamma=0$", size=14)
    p_action_params = Alice_gives_Bob(endowment_true, endowment_believed, endowment_sd, pres_target, pres_sd)[:, -1, :, 0]
    p_action_params = p_action_params / np.sum(p_action_params, axis=1, keepdims=True)
    betas_means_bob_naive = p_action_params @ betas
    im = sns.heatmap(
        p_action_params.T,
        square=True if K_values == len(A) else False,
        annot=True if K_values <= 5 else False,
        fmt=".2f",
        cmap="Greens",
        vmin=0,
        vmax=1,
        yticklabels=[f'{beta:.2f}' for beta in betas],
        xticklabels=[f'{a.name}\n{(endowment_true * payoffs(a)):.0f}' for a in A],
        cbar=False,
        ax=ax
    )
    #ax.set_ylabel("$\\beta$", size=13)
    
    ax.tick_params(axis='y', labelsize=10, labelrotation=0)
    ax.tick_params(axis='x', labelsize=10, labelrotation=0)
    ax.set_xlabel("Alice's action", size=13)

    # Create colorbar on the right side without affecting subplot sizes
    #divider = make_axes_locatable(axes[-1])
    #cax = divider.append_axes("right", size="7%", pad=0.0)
    #plt.colorbar(im.collections[0], cax=cax, label='Posterior Probability')

    ax = axes[3]
    ax.set_title(f"Posterior mean $\\beta$", size=14)

    # Plot the posterior means for each action
    df_means = pd.DataFrame({
        'Action': [a.name for a in A],
        'Posterior Mean Beta': betas_means_obs,
        'Posterior Mean Beta Bob': betas_means_bob,
        'Posterior Mean Beta Bob Naive': betas_means_bob_naive
    })
    df_means = df_means.melt(id_vars='Action', value_vars=df_means.columns[1:], var_name='Type', value_name='Mean Beta')
    palette = sns.color_palette("Blues", n_colors=1) + sns.color_palette("Reds", n_colors=1) + sns.color_palette("Greens", n_colors=1)
    sns.barplot(x='Action', y='Mean Beta', hue='Type', data=df_means, ax=ax, palette=palette)
    ax.set_ylabel("$E[\\beta|a]$", size=13)
    # Put legend at the bottom of the plot
    # Get the handles and labels from the current axes
    handles, labels = ax.get_legend_handles_labels()
    new_labels = ['Observer', 'Bob', 'Bob Naive']
    ax.legend(handles, new_labels, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, fontsize=11)

    sns.despine(ax=ax)

    fig.suptitle(f"Alice's action probabilities\n Endowment: {endowment_true}, Bob's belief: ${endowment_believed}\\pm {endowment_sd}$,  $\\rho(\\beta) = N({pres_target}, {pres_sd})$\n", size=14)
    plt.tight_layout()
    plt.show()

widgets.interactive(
    show_interactive,
    endowment_true=endowment_widget,
    endowment_believed=endowment_belief_widget,
    endowment_sd=endowment_sd_widget,
    pres_target=pres_target_widget,
    pres_sd=pres_sd_widget,
    gamma=gamma_widget
)

interactive(children=(FloatSlider(value=20.0, description="Alice's true endowment", layout=Layout(width='50%')…

In [22]:



a = np.arange(20)
print(a)
p = jax.scipy.stats.norm.pdf(a, 5, 3)
p = p/p.sum()
print(p.round(2))

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]
[0.03 0.06 0.08 0.11 0.13 0.14 0.13 0.11 0.08 0.06 0.03 0.02 0.01 0.
 0.   0.   0.   0.   0.   0.  ]


If Alice gives more that what Bob believes she has, Bob can either 

1. be happily surprised
2. realise that Alice had the potential to give more and change his view of the world

If Alice is either a rich or a poor person, Bob's realisation that she gave more than he thought she should should now make him convinced that Alice is a rich person. He would then evaluate her actions as if she was a rich person, thus making her seem less relatively generous.

In [59]:
endowment_true = 20
endowment_believed = 5
endowment_sd = 3
pres_target = 1
pres_sd = 1.5

p_action_params = Alice_gives_Bob(endowment_true, endowment_believed, endowment_sd, pres_target, pres_sd)


In [62]:
p_action_params

Array([[[[2.00000009e-04, 1.91284402e-04, 1.82831020e-04, ...,
          1.44512145e-04, 1.37630617e-04, 1.31004330e-04],
         [7.90828017e-06, 7.46702426e-06, 7.04966669e-06, ...,
          5.27988050e-06, 4.98181134e-06, 4.70011901e-06],
         [1.22969254e-07, 1.16350058e-07, 1.10083462e-07, ...,
          8.34203036e-08, 7.89105172e-08, 7.46417470e-08],
         ...,
         [3.07917454e-17, 2.91901913e-17, 2.76719394e-17, ...,
          2.11860824e-17, 2.00841206e-17, 1.90395073e-17],
         [3.61707729e-19, 3.42896349e-19, 3.25063214e-19, ...,
          2.48881379e-19, 2.35937683e-19, 2.23667104e-19],
         [4.24816133e-21, 4.02723423e-21, 3.81779476e-21, ...,
          2.92308503e-21, 2.77106757e-21, 2.62695659e-21]],

        [[4.54506255e-04, 4.42322082e-04, 4.30145039e-04, ...,
          3.69898189e-04, 3.58082500e-04, 3.46379093e-04],
         [4.92539330e-05, 4.65972007e-05, 4.40718723e-05, ...,
          3.32291675e-05, 3.13820492e-05, 2.96309991e-05],
        

In [55]:
[get_prior(endowment, endowment_believed, endowment_sd) for endowment in Endowment]

[Array(0.13298075, dtype=float32, weak_type=True),
 Array(4.9557286e-07, dtype=float32, weak_type=True)]